# Credit Risk Analysis and Assessment

This notebook focuses on analyzing credit risk patterns, default rates, and approval behaviors in our UK banking dataset.

## Analysis Goals

1. **Default Rate Analysis**
   - Analyze default rates by credit score ranges
   - Identify high-risk customer segments
   - Evaluate impact of employment status

2. **Approval Pattern Analysis**
   - Study approval rates across segments
   - Identify key approval factors
   - Analyze loan amount relationships

3. **Risk Factor Identification**
   - Determine key risk indicators
   - Analyze debt-to-income patterns
   - Study credit score impacts

4. **Portfolio Insights**
   - Segment performance analysis
   - Product risk assessment
   - Geographic risk patterns

## Expected Outcomes

1. Risk scoring model recommendations
2. Approval criteria optimization
3. Portfolio risk management strategies
4. Customer segment targeting insights

## Setup and Configuration

Import required libraries and configure the analysis environment.

In [11]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime, timedelta
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve, auc

# Visualization settings
#plt.style.use('seaborn')
%matplotlib inline

# Configure pandas display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 10)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

# Set random seed for reproducibility
np.random.seed(42)

# Define data directory
DATA_DIR = Path('C:/Users/GIDI/Desktop/Folders/REPOSITORY/FinRisk-ai/data/raw')

## Data Loading and Preprocessing

Load and prepare credit-related datasets for analysis.

In [ ]:
# Load datasets
credit_apps = pd.read_csv(DATA_DIR / 'credit_applications.csv')
bureau_data = pd.read_csv(DATA_DIR / 'credit_bureau_data.csv')
customer_data = pd.read_csv(DATA_DIR / 'customer_profiles.csv')

# Convert date columns
credit_apps['application_date'] = pd.to_datetime(credit_apps['application_date'])
customer_data['last_activity_date'] = pd.to_datetime(customer_data['last_activity_date'])

# Create analysis dataset
analysis_df = credit_apps.merge(bureau_data, on='customer_id', suffixes=('_app', '_bureau'))
analysis_df = analysis_df.merge(customer_data, on='customer_id', suffixes=('', '_profile'))

# Create derived features
analysis_df['credit_score_diff'] = analysis_df['credit_score_bureau'] - analysis_df['credit_score']
analysis_df['risk_level'] = pd.qcut(analysis_df['credit_score_bureau'], q=5, 
                                  labels=['Very High', 'High', 'Medium', 'Low', 'Very Low'])

print("Data Loading Complete!")
print("\nAnalysis Dataset Shape:", analysis_df.shape)
print("\nSample of Created Features:")
print(analysis_df[['customer_id', 'credit_score_bureau', 'credit_score', 
                  'credit_score_diff', 'risk_level']].head())

## Phase 1: Data Understanding

### 1.1 Dataset Overview
- Shape and structure of datasets
- Missing values analysis
- Data types review
- Basic statistics

### 1.2 Statistical Summaries
- Numeric variable distributions
- Key statistics for credit-related variables
- Outlier detection

### 1.3 Categorical Analysis
- Distribution of loan purposes
- Employment status breakdown
- Application status distribution
- Risk segment analysis

In [ ]:
# 1.1 Dataset Overview
def analyze_dataset(df, name):
    print(f"\n{'='*50}")
    print(f"{name} Analysis")
    print(f"{'='*50}")
    
    # Shape
    print(f"\nShape: {df.shape}")
    
    # Missing values
    missing = df.isnull().sum()
    if missing.any():
        print("\nMissing Values:")
        print(missing[missing > 0])
    else:
        print("\nNo missing values found")
    
    # Data types
    print("\nData Types:")
    print(df.dtypes)
    
    # Sample data
    print("\nSample Data:")
    display(df.head(3))

# Analyze each dataset
analyze_dataset(credit_apps, "Credit Applications")
analyze_dataset(bureau_data, "Bureau Data")
analyze_dataset(customer_data, "Customer Data")

In [ ]:
# 1.2 Statistical Summaries for Numeric Variables
def analyze_numeric_variables(df, name):
    print(f"\n{'='*50}")
    print(f"{name} Numeric Analysis")
    print(f"{'='*50}")
    
    numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
    
    # Basic statistics
    print("\nSummary Statistics:")
    display(df[numeric_cols].describe())
    
    # Distribution plots
    num_cols = len(numeric_cols)
    fig_rows = (num_cols + 2) // 3  # 3 plots per row
    
    plt.figure(figsize=(15, 5*fig_rows))
    for idx, col in enumerate(numeric_cols, 1):
        plt.subplot(fig_rows, 3, idx)
        sns.histplot(df[col], kde=True)
        plt.title(f'{col} Distribution')
        plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

# Analyze numeric variables for each dataset
analyze_numeric_variables(analysis_df, "Credit Analysis Dataset")

# Calculate correlations
plt.figure(figsize=(12, 8))
numeric_cols = analysis_df.select_dtypes(include=['int64', 'float64']).columns
correlation_matrix = analysis_df[numeric_cols].corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title('Correlation Matrix of Numeric Variables')
plt.xticks(rotation=45)
plt.yticks(rotation=45)
plt.tight_layout()
plt.show()